In [ ]:
from __future__ import (absolute_import, division,
                        print_function, unicode_literals)

import warnings
warnings.simplefilter('ignore')

# general purpose packages
import pandas as pd
import numpy as np
import os
import json
import time
import re
import csv
import subprocess
import sys

import scipy.stats as stats
import statsmodels.stats as smstats
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from dotenv import load_dotenv
from pathlib import Path

from multiprocessing import Process, Manager, Pool
import multiprocessing
from functools import partial

from collections import Counter

import seaborn as sns; sns.set()

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['backend'] = "Qt5Agg"
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

from IPython.display import display, Image

from adjustText import adjust_text
import builtins
%matplotlib inline

# for normalization
from sklearn.linear_model import QuantileRegressor

# for survival analysis
import sklearn
from sklearn import set_config

from statsmodels.regression.quantile_regression import QuantReg

# for working with yaml files
import ruamel.yaml

import itertools

In [ ]:
def get_pvalue_star(pval, thr=0.05):
    if thr == 0.05:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.05:
            return "*"
        else:
            return ""
    elif thr == 0.1:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.1:
            return "*"
        else:
            return ""

In [ ]:
# 1. Load the environment variables
load_dotenv("APA_localization.scicore.env")

# 2. Reconstruct the subdirs dictionary
subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "raw_sequencing_data_dir": os.getenv("RAW_SEQUENCING_DATA_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "wf_dir": os.getenv("WF_DIR"),
    "UCSCtracks_dir": os.getenv("UCSC_TRACKS_DIR"),
    "UCSCtracks_trackfiles_dir": os.getenv("UCSC_TRACKFILES_DIR"),
    "UCSCtracks_trackhubs_dir": os.getenv("UCSC_TRACKHUBS_DIR"),
    "human_annotation_dir": os.getenv("HUMAN_ANNOTATION_DIR"),
    "mouse_annotation_dir": os.getenv("MOUSE_ANNOTATION_DIR"),
    "shared_project_dir": os.getenv("SHARED_PROJECT_DIR"),
    "temp_dir": os.getenv("TEMP_DIR"),
    "slurm_dir": os.getenv("SLURM_DIR"),
    "slurm_scripts_dir": os.getenv("SLURM_SCRIPTS_DIR"),
    "figures_dir": os.getenv("FIGURES_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "fastq_dir": os.getenv("FASTQ_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
    "pod5_dir": os.getenv("POD5_DIR"), # nanopore-specific
    "dorado_models_dir": os.getenv("DORADO_MODELS_DIR"), # nanopore-specific
}

# 3. Reconstruct the file_paths dictionary
file_paths = {
    "human_genome_file": os.getenv("HUMAN_GENOME_FILE"),
    "human_chrom_sizes_file": os.getenv("HUMAN_CHROM_SIZES_FILE"),
    "human_annotation_file": os.getenv("HUMAN_ANNOTATION_FILE"),
    "human_basic_annotation_file": os.getenv("HUMAN_BASIC_ANNOTATION_FILE"),
    "human_polyAsite_atlas": os.getenv("HUMAN_POLYASITE_ATLAS"),
    "human_tandem_PAS": os.getenv("HUMAN_TANDEM_PAS"),
    "human_exonic_segments_gtf": os.getenv("HUMAN_EXONIC_SEGMENTS_GTF"),
    "human_exonic_segments_bed": os.getenv("HUMAN_EXONIC_SEGMENTS_BED"),
}

# 4. Safely create all subdirectories
# Using os.makedirs is highly preferred over os.system('mkdir -p')
# because it avoids opening a subshell and handles permissions gracefully in pure Python.
for path in subdirs.values():
    if path:  # Safety check to ensure the variable was actually found in the .env
        os.makedirs(path, exist_ok=True)

print("Environment loaded and directories verified.")

# Prepare start samples for nanoflowz

In [ ]:
# we extract the paths to .pod5 files

os.system("""find """+subdirs['pod5_dir']+""" -name '*.pod5' > """+subdirs['temp_dir']+"""pod5_files.tsv""")

In [ ]:
pod5_file_paths = pd.read_csv(subdirs['temp_dir']+'pod5_files.tsv',delimiter="\t",
                                   index_col=None,header=None)
pod5_file_paths.columns = ['pod5']
pod5_file_paths['sample_id'] = pod5_file_paths.apply(lambda x:x['pod5'].split('/')[-2],1)

# we are only interested in barcodes 01-12
sel_sample_ids = range(1,13)
sel_sample_ids = [('barcode0'+str(elem) if len(str(elem))==1 else 'barcode'+str(elem)) for elem in sel_sample_ids]
pod5_file_paths = pod5_file_paths.loc[pod5_file_paths['sample_id'].isin(sel_sample_ids)].reset_index(drop=True)

In [ ]:
WF_version = 'v1p4_ONT_FACSorganelles_Aleksei_v1' # v1p4 at the beginning indicated Dorado version, while v1 at the end indicates the particular version of the analysis

dir_path = subdirs['wf_runs_dir']+WF_version+'/'
input_for_this_WF_version = dir_path+'input/pod5/'

pod5_file_paths['start_file_path'] = pod5_file_paths.apply(lambda x:input_for_this_WF_version+x['sample_id']+'/'+Path(x['pod5']).name,1)

# uncomment if need to re-generate, otherwise re-creation may invoke unnecessary nextflow re-execution
for index, row in pod5_file_paths.iterrows():
    new_pod5_path_dir = Path(row['start_file_path']).parent
    new_pod5_path_dir.mkdir(parents=True, exist_ok=True)

    command = 'rm -f '+row['start_file_path']+' && ln -f -s '+row['pod5']+' '+row['start_file_path']
    out = subprocess.check_output(command, shell=True)
pod5_file_paths = pod5_file_paths.drop(['pod5'],axis=1).rename(columns={'start_file_path':'pod5'})

pod5_file_paths[['sample_id','pod5']].to_csv(dir_path+'start_samples.tsv', sep=str('\t'),header=True,index=None,quoting=csv.QUOTE_NONE)

# executing nanoflowz

We've created a conda env "nextflow" to execute nanoflowz:

```bash
conda create --name nextflow bioconda::nextflow
conda activate nextflow
```

# Drafts

In [ ]:
# mamba create -c conda-forge -c bioconda -n nanopore python=3.12
# mamba activate nanopore
# mamba install bioconda::lib-pod5 conda-forge::ipykernel conda-forge::pandas conda-forge::scipy conda-forge::seaborn conda-forge::scikit-learn bioconda::samtools bioconda::pysam bioconda::htseq bioconda::bedtools -y
# pip install pod5

In [ ]:
subdirs['dorado_models_dir']

In [ ]:
# cd to dorado_models_dir in the login node
# conda activate nanopore
command = "dorado download --model dna_r10.4.1_e8.2_400bps_sup@v5.2.0 --models-directory "+subdirs['dorado_models_dir']
command

In [ ]:
# cd to dorado_models_dir in the login node
# conda activate nanopore
command = "dorado download --model dna_r10.4.1_e8.2_5khz_stereo@v1.4 --models-directory "+subdirs['dorado_models_dir']
command

Run one file, just to check

In [ ]:
# srun --nodes=1 --cpus-per-task=12 --mem=80G --qos=a100-30min --partition=a100 --time=0:29:00 --gres=gpu:1 --pty bash     good
# srun --nodes=1 --cpus-per-task=12 --mem=80G --qos=a100-30min --partition=a100-80g --time=0:29:00 --gres=gpu:1 --pty bash     good

# srun --nodes=1 --cpus-per-task=12 --mem=80G --qos=h200-30min --partition=h200 --time=0:29:00 --gres=gpu:1 --pty bash     good
# srun --nodes=1 --cpus-per-task=12 --mem=80G --qos=titan-30min --partition=titan --time=0:29:00 --gres=gpu:1 --pty bash   works, but very slow
# srun --nodes=1 --cpus-per-task=12 --mem=80G --qos=rtx4090-30min --partition=rtx4090 --time=0:29:00 --gres=gpu:1 --pty bash      good, may be slower than for h200   
# srun --nodes=1 --cpus-per-task=12 --mem=80G --qos=l40s-30min --partition=l40s --time=0:29:00 --gres=gpu:1 --pty bash     should also work

In [ ]:
# check CUDA driver version (after module load CUDA)
# nvidia-smi

In [ ]:
threads = 12
outdir = subdirs['wf_runs_dir']+'v1p4_ONT_FACSorganelles/'
command = 'mkdir -p '+outdir+'; conda activate nanopore; module load CUDA; dorado basecaller '+\
subdirs['dorado_models_dir']+'dna_r10.4.1_e8.2_400bps_sup@v5.2.0 '+\
subdirs['pod5_dir']+'barcode01/PBI81495_pass_barcode01_44ac0a2d_69015063_0.pod5'+\
' --estimate-poly-a --poly-a-config '+subdirs['toy_dir']+'polya_config.toml'+\
' --mm2-opts "-x splice -Y" --reference '+file_paths['human_genome_file']+\
' | '+\
'samtools sort -@ '+str(threads)+' -o '+outdir+'PBI81495_pass_barcode01_44ac0a2d_69015063_0.sorted.bam && samtools index -@ '+str(threads)+' '+outdir+'PBI81495_pass_barcode01_44ac0a2d_69015063_0.sorted.bam'
command

In [ ]:
# try with no trimming

threads = 12
outdir = subdirs['wf_runs_dir']+'v1p4_ONT_FACSorganelles_with_Trimming/'
command = 'mkdir -p '+outdir+'; conda activate nanopore; module load CUDA; dorado basecaller --no-trim '+\
subdirs['dorado_models_dir']+'dna_r10.4.1_e8.2_400bps_sup@v5.2.0 '+\
subdirs['pod5_dir']+'barcode01/PBI81495_pass_barcode01_44ac0a2d_69015063_0.pod5'+\
' --estimate-poly-a --poly-a-config '+subdirs['toy_dir']+'polya_config.toml'+\
' --mm2-opts "-x splice -Y" --reference '+file_paths['human_genome_file']+\
' | '+\
' samtools sort -@ '+str(threads)+' -o '+outdir+'PBI81495_pass_barcode01_44ac0a2d_69015063_0.sorted.bam && samtools index -@ '+str(threads)+' '+outdir+'PBI81495_pass_barcode01_44ac0a2d_69015063_0.sorted.bam'
command

In [ ]:
# try with explicit trimming

threads = 12
outdir = subdirs['wf_runs_dir']+'v1p4_ONT_FACSorganelles_with_Trimming_v2/'
command = 'mkdir -p '+outdir+'; conda activate nanopore; module load CUDA; dorado basecaller --kit-name SQK-PCB114-24 --trim all '+\
subdirs['dorado_models_dir']+'dna_r10.4.1_e8.2_400bps_sup@v5.2.0 '+\
subdirs['pod5_dir']+'barcode01/PBI81495_pass_barcode01_44ac0a2d_69015063_0.pod5'+\
' --estimate-poly-a --poly-a-config '+subdirs['toy_dir']+'polya_config.toml'+\
' --mm2-opts "-x splice -Y" --reference '+file_paths['human_genome_file']+\
' | '+\
'samtools sort -@ '+str(threads)+' -o '+outdir+'PBI81495_pass_barcode01_44ac0a2d_69015063_0.sorted.bam && samtools index -@ '+str(threads)+' '+outdir+'PBI81495_pass_barcode01_44ac0a2d_69015063_0.sorted.bam'
command

In [ ]:
# at MAPQ>50
anchor = 20.6
vals = [anchor, 25.4, 28.3, 27.9]
[np.round(elem/anchor,2)/1.37 for elem in vals]

In [ ]:
# at MAPQ>1
anchor = 23.6
vals = [anchor, 29.3, 32.4, 32.4, 12.4, 29.9]
[np.round(elem/anchor,2)/1.37 for elem in vals]

In [ ]:
65*0.72,95*0.9

In [ ]:
s = """gttttgggggagaactggaaagccgagggtagccgagcggggcgggcgct
ctggagcggcgggtgctcgggctgccgtccgctccgccagaagcaccgag
cagccgagccggggcccgccgccctcctcctccatgaggcccgagtgagg
cgcggcggctatagccgacccgcggcgccttccccccgcgtcctatcgcg
agcgcagcggcagcggcccctggaggaggaggcggaggaggaggagcATG
TCGGACGGTTTCGATCGGGCCCCAGAGCAAACGAGGCCCCTGAGAGCTCC
ACCTAGTTCACAGGATAAAATCCCACAGCAGAACTCGGAGTCAGCAATGG
CTAAGCCCCAGGTGGTTGTAGCTCCTGTATTAATGTCTAAGCTGTCTGTG
AATGCCCCTGAATTTTACCCTTCAGGTTATTCTTCCAGTTACACAGAATC
CTATGAGGATGGTTGTGAGGATTATCCTACTCTATCAGAATATGTTCAGG
ATTTTTTGAATCATCTTACAGAGCAGCCTGGCAGTTTTGAAACTGAAATT
GAACAGTTTGCAGAGACCCTGAATGGTTGTGTTACAACAGATGATGCTTT
GCAAGAACTTGTGGAACTCATCTATCAACAGGCCACATCTATCCCAAATT
TCTCTTATATGGGAGCTCGCCTGTGTAATTACCTGTCCCATCATCTGACA
ATTAGCCCACAGAGTGGCAACTTCCGCCAATTGCTACTTCAAAGATGTCG
GACTGAATATGAAGTTAAAGATCAAGCTGCAAAAGGGGATGAAGTTACTC
GAAAACGATTTCATGCATTTGTACTCTTTCTGGGAGAACTTTATCTTAAC
CTGGAGATCAAGGGAACAAATGGACAGGTTACAAGAGCAGATATTCTTCA
GGTTGGTCTTCGAGAATTGCTGAATGCCCTGTTTTCTAATCCTATGGATG
ACAATTTAATTTGTGCAGTAAAATTGTTAAAGTTGACAGGATCAGTTTTG
GAAGATGCTTGGAAGGAAAAAGGAAAGATGGATATGGAAGAAATTATTCA
GAGAATTGAAAACGTTGTCCTAGATGCAAACTGCAGTAGAGATGTAAAAC
AGATGCTCTTGAAGCTTGTAGAACTCCGGTCAAGTAACTGGGGCAGAGTC
CATGCAACTTCAACATATAGAGAAGCAACACCAGAAAATGATCCTAACTA
CTTTATGAATGAACCAACATTTTATACATCTGATGGTGTTCCTTTCACTG
CAGCTGATCCAGATTACCAAGAGAAATACCAAGAATTACTTGAAAGAGAG
GACTTTTTTCCAGATTATGAAGAAAATGGAACAGATTTATCCGGGGCTGG
TGATCCATACTTGGATGATATTGATGATGAGATGGACCCAGAGATAGAAG
AAGCTTATGAAAAGTTTTGTTTGGAATCAGAGCGTAAGCGAAAACAGTAA
agttaaatttcagcatatcagttttataaagcagtttaggtatggtgatt
tagcagaacacaagagagcaagaaaatgtcacatctataccaaattaagg
atgttgagttatgttactaatgtatgcaactttaattttgtttaacacta
tctgccaaaataaactttattccctataacttaaaatgtgtatatatata
taatagtttattatgtacagttaattctactgttttggctgcaataaaat
cgattttgaaataaatgaaatgttgaaaattttg"""
len(s)

In [ ]:
1768-500

In [ ]:
# try duplex basecalling

threads = 12
outdir = subdirs['wf_runs_dir']+'v1_ONT_FACSorganelles/'
command = 'mkdir -p '+outdir+'; module load CUDA; dorado duplex '+\
subdirs['dorado_models_dir']+'dna_r10.4.1_e8.2_5khz_stereo@v1.4 '+\
subdirs['pod5_dir']+'barcode01/PBI81495_pass_barcode01_44ac0a2d_69015063_0.pod5'+\
' --mm2-opts "-x splice -Y" --reference '+file_paths['human_genome_file']+\
' | '+\
'samtools sort -@ '+str(threads)+' -o '+outdir+'PBI81495_pass_barcode01_44ac0a2d_69015063_0.duplex.sorted.bam && samtools index -@ '+str(threads)+' '+outdir+'PBI81495_pass_barcode01_44ac0a2d_69015063_0.duplex.sorted.bam'
command

In [ ]:
# try duplex basecalling

threads = 12
outdir = subdirs['wf_runs_dir']+'v1_ONT_FACSorganelles/'
command = 'mkdir -p '+outdir+'; module load CUDA; dorado duplex '+\
subdirs['dorado_models_dir']+'dna_r10.4.1_e8.2_400bps_sup@v5.2.0 '+\
subdirs['pod5_dir']+'barcode01/PBI81495_pass_barcode01_44ac0a2d_69015063_0.pod5'+\
' --mm2-opts "-x splice -Y" --reference '+file_paths['human_genome_file']+\
' | '+\
'samtools sort -@ '+str(threads)+' -o '+outdir+'PBI81495_pass_barcode01_44ac0a2d_69015063_0.duplex.sorted.bam && samtools index -@ '+str(threads)+' '+outdir+'PBI81495_pass_barcode01_44ac0a2d_69015063_0.duplex.sorted.bam'
command

In [ ]:
WF_version = 'v2_ONT'
general_output_dir = subdirs['wf_runs_dir']+WF_version+'/output/'
bam_output = general_output_dir+'bam_small_files/'

os.system('mkdir -p '+bam_output) # create all subdirs

a = []
for index, row in metadata.iterrows():
    dir_to_search = subdirs['pod5_dir']+row['bio_id']+'/'
    os.system("""find """+dir_to_search+""" -name '*.pod5' > """+subdirs['temp_dir']+row['bio_id']+"""_small_pod5_files.tsv""")
    tmp_files_df = pd.read_csv(subdirs['temp_dir']+row['bio_id']+"""_small_pod5_files.tsv""",delimiter="\t",index_col=None,header=None)
    tmp_files_df.columns = ['path']
    tmp_files_df['bio_id'] = row['bio_id']
    tmp_files_df['sample'] = row['sample']
    a.append(tmp_files_df)
small_pod5_files_df = pd.concat(a).reset_index(drop=True)

small_pod5_files_df['small_file_id'] = small_pod5_files_df.apply(lambda x: x['path'].split('/')[-1].split('.pod5')[0],1)
small_pod5_files_df['id'] = small_pod5_files_df.index+1
small_pod5_files_df['out_sorted_bam'] = bam_output+small_pod5_files_df['sample']+'.'+small_pod5_files_df['small_file_id']+'.sorted.bam'

table_of_all_small_pod5_files = subdirs['temp_dir']+'all_small_pod5_files.tsv'
small_pod5_files_df[['id','path','out_sorted_bam']].to_csv(table_of_all_small_pod5_files, sep=str('\t'),header=False,index=None)

threads,RAM = 6,40

sbatch_script_path = subdirs['slurm_scripts_dir']+'Dorado_SUP_basecall.sbatch'
f = open(sbatch_script_path,'w')
preambula = \
"""#!/bin/bash

#SBATCH --job-name=dorado_SUP
#SBATCH --time=0:29:00
#SBATCH --qos=gpu30min
#SBATCH --partition=a100
#SBATCH --gres=gpu:1
#SBATCH --output="""+subdirs['slurm_dir']+"""%A_%a.out
#SBATCH --error="""+subdirs['slurm_dir']+"""%A_%a.err
#SBATCH --cpus-per-task="""+str(threads)+"""    #Number of cores to reserve
#SBATCH --mem="""+str(RAM)+"""G     #Amount of RAM/core to reserve
#SBATCH --array=1-"""+str(len(small_pod5_files_df))+"""%150

source ~/.bashrc
module load CUDA
conda activate nanopore
"""
f.write(preambula+'\n')
f.write("""pod5=$(awk -v i=$SLURM_ARRAY_TASK_ID '{ if ($1 == i) { print $2} }' """+table_of_all_small_pod5_files+')\n')
f.write("""out_sorted_bam=$(awk -v i=$SLURM_ARRAY_TASK_ID '{ if ($1 == i) { print $3} }' """+table_of_all_small_pod5_files+')\n')
f.write("""echo $out_bam"""+'\n')

command = 'dorado basecaller '+\
subdirs['dorado_models_dir']+'rna004_130bps_sup@v5.1.0 '+\
'$pod5'+\
' --estimate-poly-a --poly-a-config '+subdirs['toy_dir']+'polya_config.toml'+\
' --mm2-opts "-x splice -Y" --reference '+file_paths['mouse_genome_file']+\
' | '+\
'samtools sort -@ '+str(threads)+' -o $out_sorted_bam && samtools index -@ '+str(threads)+' $out_sorted_bam'

f.write(command)
f.close()

print('sbatch '+sbatch_script_path)